# ARGUS · Session 2 — Phase B Fine-tune + Hard-Example Mining

## Before starting — attach these two datasets

| Step | What to do |
|------|------------|
| 1 | **Attach your S1 checkpoint**: Upload `argus_s1.pt` as a Kaggle dataset (or use the `argus-s1` dataset from a previous session). The notebook will find it automatically in `/kaggle/input/`. |
| 2 | **Attach IDD**: Search Kaggle → `abhishekprajapat/idd-20k` → Add to notebook. The IDD cell auto-detects the mount point regardless of slug name. |
| 3 | *(Optional)* Set `KAGGLE_USERNAME` + `KAGGLE_KEY` in **Add-ons → Secrets** to also download BDD100K and UA-DETRAC via API. |

Run the **Pre-flight** cell (cell 3) first — it shows exactly which datasets are found and which are missing.

---

## What this notebook does (~30 h on T4×2)

| Phase | Description | Expected |
|-------|-------------|----------|
| **Phase B** | Full-network fine-tune, 20 epochs on merged dataset | mAP50 0.659→0.76–0.82 |
| **Hard mining** | Score images on 5 proximity signals, select top 70% | |
| **HM-A** | 6 epochs high-LR on hard subset | |
| **HM-B** | Re-score bottom 10%, 6 epochs reduced-LR | |
| **Output** | `argus_s2.pt` — download from Output tab for Session 3 |

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
from pathlib import Path
import torch, os, json as _json

NC      = 5
CLASSES = ['car', 'motorcycle', 'bus', 'truck', 'bicycle']

WORK     = Path('/kaggle/working')
OUT_DIR  = WORK / 'argus_data'
MERGED   = OUT_DIR / 'merged'
HARD_DIR = OUT_DIR / 'hard'
HELD_DIR = OUT_DIR / 'held'
RUNS_DIR = WORK / 'runs'
INPUT    = Path('/kaggle/input')
yaml_path = MERGED / 'data.yaml'

for d in [MERGED/'train/images', MERGED/'train/labels',
          MERGED/'valid/images', MERGED/'valid/labels',
          HARD_DIR/'train/images', HARD_DIR/'train/labels',
          HARD_DIR/'valid/images', HARD_DIR/'valid/labels',
          HELD_DIR/'valid/images', HELD_DIR/'valid/labels',
          RUNS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

UPLOADED_S1 = WORK / 'argus_s1.pt'

# ── Locate argus_s1.pt — attached dataset or manual upload ───────────────────
if not UPLOADED_S1.exists():
    hit = next(INPUT.rglob('argus_s1.pt'), None)
    if hit:
        import shutil as _sh
        _sh.copy2(hit, UPLOADED_S1)
        print(f'Copied argus_s1.pt from {hit}')
    else:
        print('⚠  argus_s1.pt not found — attach it as a Kaggle dataset or upload manually')
else:
    print(f'argus_s1.pt ready at {UPLOADED_S1}')

# Phase B — full fine-tune on merged dataset
PHB_BEST = RUNS_DIR/'s2_phb'/'argus_s2_phb'/'weights'/'best.pt'
PHB_LAST = RUNS_DIR/'s2_phb'/'argus_s2_phb'/'weights'/'last.pt'

# Hard-mining Phase A (high LR)
HMA_BEST = RUNS_DIR/'s2_hma'/'argus_s2_hma'/'weights'/'best.pt'
HMA_LAST = RUNS_DIR/'s2_hma'/'argus_s2_hma'/'weights'/'last.pt'

# Hard-mining Phase B (reduced LR)
HMB_BEST = RUNS_DIR/'s2_hmb'/'argus_s2_hmb'/'weights'/'best.pt'
HMB_LAST = RUNS_DIR/'s2_hmb'/'argus_s2_hmb'/'weights'/'last.pt'

DEVICE = ','.join(str(i) for i in range(torch.cuda.device_count())) or 'cpu'
BATCH  = 16
IMGSZ  = 640

KGL_USER = os.environ.get('KAGGLE_USERNAME','')
KGL_KEY  = os.environ.get('KAGGLE_KEY','')
if KGL_USER and KGL_KEY:
    kdir = Path.home()/'.kaggle'; kdir.mkdir(exist_ok=True)
    (kdir/'kaggle.json').write_text(_json.dumps({'username':KGL_USER,'key':KGL_KEY}))
    (kdir/'kaggle.json').chmod(0o600)
    print(f'Kaggle creds: {KGL_USER}')
else:
    print('Kaggle creds NOT set — BDD100K + UA-DETRAC will be skipped')

print(f'DEVICE={DEVICE}  BATCH={BATCH}  IMGSZ={IMGSZ}')

In [ ]:
# ── Kaggle API credentials — paste here if Secrets don't work ────────────────
# Get your key from https://kaggle.com/settings -> API -> Create New Token
import os, json
from pathlib import Path

KAGGLE_USER = ''   # <- paste your Kaggle username
KAGGLE_KEY  = ''   # <- paste your API key

# Fall back to environment Secrets if not hardcoded
if not KAGGLE_USER: KAGGLE_USER = os.environ.get('KAGGLE_USERNAME', '')
if not KAGGLE_KEY:  KAGGLE_KEY  = os.environ.get('KAGGLE_KEY', '')

if KAGGLE_USER and KAGGLE_KEY:
    kdir = Path.home() / '.kaggle'
    kdir.mkdir(exist_ok=True)
    (kdir / 'kaggle.json').write_text(json.dumps({'username': KAGGLE_USER, 'key': KAGGLE_KEY}))
    (kdir / 'kaggle.json').chmod(0o600)
    print(f'Kaggle: authenticated as {KAGGLE_USER}')
else:
    print('Kaggle: no credentials -- API downloads disabled, using attached datasets only')


In [ ]:
# ── Install + GPU check ──────────────────────────────────────────────────────
import subprocess, sys, torch

n_gpu = torch.cuda.device_count()
print(f'PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()} | GPUs: {n_gpu}')
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name}  {p.total_memory // 1024**2} MB')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'ultralytics>=8.4.0', 'pyyaml', 'pycocotools', 'kaggle'], check=True)
import ultralytics; ultralytics.checks()

In [ ]:
# ── Pre-flight: what's in /kaggle/input/ ──────────────────────────────────────
import os
from pathlib import Path

def _quick_count(d, ext=None, limit=200):
    """Count files up to limit (fast — avoids multi-minute scans on huge datasets)."""
    n = 0
    for p in d.rglob('*' if ext is None else f'*{ext}'):
        if p.is_file(): n += 1
        if n >= limit: return f'{limit}+'
    return str(n)

print('Mounted datasets in /kaggle/input/datasets/:')
ds_root = INPUT / 'datasets'
mounted_names = set()
if ds_root.exists():
    for user_dir in sorted(ds_root.iterdir()):
        if not user_dir.is_dir(): continue
        print(f'  {user_dir.name}/')
        mounted_names.add(user_dir.name.lower())
        for ds_dir in sorted(user_dir.iterdir()):
            if not ds_dir.is_dir(): continue
            try:
                mb = sum(f.stat().st_size for f in ds_dir.rglob('*') if f.is_file()) // 1024**2
                print(f'    {ds_dir.name}/  (~{mb} MB)')
            except Exception:
                print(f'    {ds_dir.name}/')
elif INPUT.exists():
    for d in sorted(INPUT.iterdir()):
        if d.is_dir():
            mounted_names.add(d.name.lower())
            print(f'  {d.name}/')

print()
s1_ok  = UPLOADED_S1.exists() or next(INPUT.rglob('argus_s1.pt'), None) is not None
idd_ok = any('idd' in n for n in mounted_names) or next(INPUT.rglob('idd20k_final'), None) is not None
bdd_ok = any('bdd' in n for n in mounted_names) or bool(KGL_USER)
print(f'  argus_s1.pt : {"OK" if s1_ok  else "MISSING"}')
print(f'  IDD dataset : {"OK" if idd_ok  else "MISSING -- attach abhishekprajapat/idd-20k"}')
print(f'  BDD100K     : {"OK" if bdd_ok else "MISSING -- attach any bdd100k dataset or set Kaggle creds"}')
print()
if not s1_ok or not idd_ok:
    print('WARNING: required datasets missing')
else:
    print('Required datasets look OK -- proceed')


In [ ]:
# ── UA-DETRAC → YOLO (overhead CCTV) ─────────────────────────────────────────
# Adds overhead traffic camera viewpoint — critical for near-miss scenarios.
# car→0  bus→2  van/truck→3  motorbike→1
# 2k held-out images locked per session (random.seed(42) — deterministic).
import subprocess, os, re, random, shutil as _shutil, xml.etree.ElementTree as ET
from pathlib import Path
from tqdm import tqdm

def _imglink(src, dst):
    try: os.link(src, dst)
    except OSError: os.symlink(Path(src).resolve(), dst)

random.seed(42)
FLAG = OUT_DIR / '.uadetrac_done'
if FLAG.exists():
    print('UA-DETRAC: already done — skipping')
else:
    DETRAC_MAP = {'car':0,'van':3,'truck':3,'bus':2,'motorbike':1,'motorcycle':1,'others':0}
    IMG_W, IMG_H = 960, 540  # UA-DETRAC standard resolution

    DETRAC_ROOT = next((INPUT/s for s in ['ua-detrac','uadetrac','ua_detrac','detrac-training']
                        if (INPUT/s).exists()), None)
    # Also search nested datasets/<user>/<ds>/ (Kaggle merged dataset mounting)
    if DETRAC_ROOT is None:
        ds_root = INPUT / 'datasets'
        if ds_root.exists():
            for user_dir in sorted(ds_root.iterdir()):
                if not user_dir.is_dir() or DETRAC_ROOT: break
                for ds_dir in sorted(user_dir.iterdir()):
                    if not ds_dir.is_dir(): continue
                    # UA-DETRAC: XML files + images in MVI_* sequence dirs
                    xml_count = sum(1 for _ in ds_dir.rglob('*.xml'))
                    if xml_count > 50:
                        DETRAC_ROOT = ds_dir
                        print(f'UA-DETRAC: found at datasets/{user_dir.name}/{ds_dir.name}/ ({xml_count} XMLs)')
                        break
    if DETRAC_ROOT is None and KGL_USER:
        RAW = WORK/'_detrac_raw'; RAW.mkdir(parents=True, exist_ok=True)
        dl_flag = RAW/'.downloaded'
        SLUGS = ['datatang/ua-detrac','dtrackers/ua-detrac','fanbyprince/ua-detrac',
                 'robustchicken/ua-detrac','adamdodge/ua-detrac']
        if dl_flag.exists():
            print('UA-DETRAC: already downloaded'); DETRAC_ROOT = RAW
        else:
            for slug in SLUGS:
                r = subprocess.run(['kaggle','datasets','download','-d',slug,
                                    '-p',str(RAW),'--unzip'], capture_output=True, text=True)
                if r.returncode == 0:
                    dl_flag.touch(); DETRAC_ROOT = RAW
                    print(f'  Downloaded via {slug}'); break
                print(f'  {slug}: {r.stderr.strip()[:100]}')
            else:
                print('  All UA-DETRAC slugs failed — skipping')

    if DETRAC_ROOT is None:
        print('UA-DETRAC: not found — set Kaggle creds or attach dataset')
    else:
        # Build seq_name → XML map (UA-DETRAC: one XML per sequence, e.g. MVI_20011.xml)
        seq_xmls = sorted(DETRAC_ROOT.rglob('*.xml'))
        xml_by_seq = {x.stem: x for x in seq_xmls}
        print(f'Found {len(seq_xmls)} sequence XMLs')

        # Parse all XMLs into memory: seq_name → {frame_num → [(ac,cx,cy,nw,nh)]}
        seq_ann = {}
        for xp in tqdm(seq_xmls, desc='Parse XMLs'):
            try: root_el = ET.parse(xp).getroot()
            except ET.ParseError: continue
            frames = {}
            for frame in root_el.findall('.//frame'):
                fn = int(frame.get('num', 0))
                boxes = []
                for tgt in frame.findall('.//target'):
                    attr = tgt.find('attribute')
                    vtype = (attr.get('vehicle_type','') if attr is not None else '').lower()
                    ac = DETRAC_MAP.get(vtype)
                    if ac is None: continue
                    box = tgt.find('box')
                    if box is None: continue
                    try:
                        l = float(box.get('left',0)); t = float(box.get('top',0))
                        bw = float(box.get('width',0)); bh = float(box.get('height',0))
                    except ValueError: continue
                    if bw <= 0 or bh <= 0: continue
                    cx = max(.001, min(.999, (l+bw/2)/IMG_W))
                    cy = max(.001, min(.999, (t+bh/2)/IMG_H))
                    nw = max(.001, min(.999, bw/IMG_W))
                    nh = max(.001, min(.999, bh/IMG_H))
                    boxes.append((ac, cx, cy, nw, nh))
                if boxes: frames[fn] = boxes
            if frames: seq_ann[xp.stem] = frames

        # Find all images
        all_imgs = sorted(DETRAC_ROOT.rglob('img*.jpg')) + sorted(DETRAC_ROOT.rglob('img*.png'))
        imgs_shuffled = list(all_imgs); random.shuffle(imgs_shuffled)
        held_set = set(str(p) for p in imgs_shuffled[:2000])

        HELD_DIR.mkdir(parents=True, exist_ok=True)
        (HELD_DIR/'valid'/'images').mkdir(parents=True, exist_ok=True)
        (HELD_DIR/'valid'/'labels').mkdir(parents=True, exist_ok=True)

        nv = n_held = 0
        for img in tqdm(all_imgs, desc='UA-DETRAC'):
            seq_name = img.parent.name
            m = re.search(r'(\d+)', img.stem[-8:])
            fn = int(m.group(1)) if m else 0
            ann = seq_ann.get(seq_name, {}).get(fn)
            if not ann: continue
            lines = [f'{ac} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}' for ac,cx,cy,nw,nh in ann]
            stem = f'detrac_{seq_name}_{img.stem}'
            if str(img) in held_set:
                hi = HELD_DIR/'valid'/'images'; hl = HELD_DIR/'valid'/'labels'
                link = hi/(stem+img.suffix)
                if not link.exists(): _imglink(img, link)
                (hl/(stem+'.txt')).write_text('\n'.join(lines))
                n_held += 1
            else:
                dst = 'train' if random.random() < 0.9 else 'valid'
                di = MERGED/dst/'images'; dl = MERGED/dst/'labels'
                link = di/(stem+img.suffix)
                if not link.exists(): _imglink(img, link)
                (dl/(stem+'.txt')).write_text('\n'.join(lines))
                nv += 1

        (HELD_DIR/'data.yaml').write_text(
            f'path: {HELD_DIR.resolve()}\nval: valid/images\nnc: {NC}\nnames: {CLASSES}\n')
        print(f'UA-DETRAC: {nv:,} train/val | {n_held:,} held-out locked')
        FLAG.touch(); print('UA-DETRAC: done')
    if DETRAC_ROOT is not None and str(DETRAC_ROOT).startswith(str(WORK)):
        _shutil.rmtree(DETRAC_ROOT, ignore_errors=True)
        print(f'  Freed {DETRAC_ROOT}')

In [ ]:
# ── BDD100K → YOLO ───────────────────────────────────────────────────────────
# motorcycle ×3 oversample, bicycle ×2 oversample for class balance.
# Checks /kaggle/input/ first (attached dataset), falls back to Kaggle API.
import shutil as _shutil

def _imglink(src: 'Path', dst: 'Path'):
    # Hardlink if same filesystem (deletable raw), symlink if cross-fs (input/).
    try: os.link(src, dst)
    except OSError: os.symlink(src.resolve(), dst)
import os, subprocess, yaml
from pathlib import Path
from tqdm import tqdm

FLAG = OUT_DIR / '.bdd_done'
if FLAG.exists():
    print('BDD100K: already done — skipping')
else:
    name_to_argus = {'car':0,'automobile':0,'motorcycle':1,'motorbike':1,'motor':1,
                     'bus':2,'truck':3,'van':3,'bicycle':4,'bike':4}

    # ── Locate BDD100K — attached dataset OR API download ────────────────────
    BDD_DIR = None

    # 1. Check /kaggle/input/ for an attached dataset
    ATTACHED_SLUGS = ['bdd100k-yolo-format','bdd100k-dataset','bdd100k','bdd100k-yolo',
                      'bdd100k-yolo-v8','bdd-100k','bdd100k_yolo','bdd_100k','bdd100k_yolo_format']
    for slug in ATTACHED_SLUGS:
        candidate = INPUT / slug
        if candidate.exists() and any(candidate.rglob('*.jpg')):
            BDD_DIR = candidate
            print(f'BDD100K: found attached dataset at {BDD_DIR}')
            break

    # 2. Nested datasets/<user>/<ds>/ search (Kaggle merges user datasets this way)
    if BDD_DIR is None:
        ds_root = INPUT / 'datasets'
        if ds_root.exists():
            for user_dir in sorted(ds_root.iterdir()):
                if not user_dir.is_dir() or BDD_DIR: break
                for ds_dir in sorted(user_dir.iterdir()):
                    if not ds_dir.is_dir(): continue
                    # Check data.yaml for BDD class names (rider, traffic light, traffic sign)
                    for yf in ds_dir.rglob('data.yaml'):
                        try:
                            cfg = yaml.safe_load(yf.read_text())
                            names_list = cfg.get('names', [])
                            if isinstance(names_list, dict): names_list = list(names_list.values())
                            bdd_markers = {'rider', 'traffic light', 'traffic sign', 'motor'}
                            if bdd_markers & {n.strip().lower() for n in names_list}:
                                BDD_DIR = ds_dir
                                print(f'BDD100K: found at datasets/{user_dir.name}/{ds_dir.name}/ (BDD class names)')
                                break
                        except Exception: pass
                    if BDD_DIR: break
                    # Fallback: largest unlabelled dataset (>50k jpgs, not IDD)
                    if not BDD_DIR and 'idd' not in str(ds_dir).lower() and 'argus' not in str(ds_dir).lower():
                        jpg_n = sum(1 for _ in ds_dir.rglob('*.jpg'))
                        if jpg_n > 50000:
                            BDD_DIR = ds_dir
                            print(f'BDD100K: found at datasets/{user_dir.name}/{ds_dir.name}/ ({jpg_n} images, assumed BDD)')
                            break

    # 3. Fallback: API download
    if BDD_DIR is None and KGL_USER:
        RAW = WORK / '_bdd_raw'; RAW.mkdir(parents=True, exist_ok=True)
        dl_flag = RAW / '.downloaded'
        API_SLUGS = ['farzadnekouei/bdd100k-yolo-format','awsaf49/bdd100k-dataset',
                     'a2zcode/bdd100k','gautamchettiar/bdd100k','a7madmostafa/bdd100k-yolo']
        if dl_flag.exists():
            print('BDD100K: already downloaded'); BDD_DIR = RAW
        else:
            for slug in API_SLUGS:
                r = subprocess.run(['kaggle','datasets','download','-d',slug,
                                    '-p',str(RAW),'--unzip'], capture_output=True, text=True)
                if r.returncode == 0:
                    dl_flag.touch(); BDD_DIR = RAW
                    print(f'  Downloaded via {slug}'); break
                print(f'  {slug}: {r.stderr.strip()[:120]}')
            if BDD_DIR is None:
                print('  All BDD100K slugs failed — skipping')

    if BDD_DIR is None:
        print('BDD100K: not found — attach dataset or set Kaggle Secrets')
    else:
        # Auto-detect class map from data.yaml
        bdd_map = None
        for yf in sorted(BDD_DIR.rglob('*.yaml')):
            try:
                cfg = yaml.safe_load(yf.read_text())
                names = cfg.get('names', [])
                if isinstance(names, dict): names = [names[k] for k in sorted(names)]
                m = {i: name_to_argus[n.lower().strip()] for i, n in enumerate(names)
                     if n.lower().strip() in name_to_argus}
                if m: bdd_map = m; print(f'  BDD map (from {yf.name}): {m}'); break
            except Exception: pass
        if bdd_map is None:
            bdd_map = {0:0, 1:1, 2:2, 7:3, 8:4, 10:1}
            print(f'  BDD map: fallback {bdd_map}')

        for split, dst in [('train','train'), ('val','valid')]:
            idirs = [d for d in BDD_DIR.rglob('images') if split in str(d)]
            ldirs = [d for d in BDD_DIR.rglob('labels') if split in str(d)]
            if not idirs: print(f'  BDD {split}: not found — skip'); continue
            idir = idirs[0]
            ldir = ldirs[0] if ldirs else idir.parent.parent/'labels'/split
            di = MERGED/dst/'images'; dl = MERGED/dst/'labels'
            nv = nm = nb_cnt = 0
            for img in tqdm(list(idir.glob('*.*')), desc=f'BDD {split}'):
                lp = ldir / (img.stem + '.txt')
                if not lp.exists(): continue
                lines, has_m, has_b = [], False, False
                for row in lp.read_text().strip().splitlines():
                    p = row.split()
                    if not p: continue
                    try: orig = int(p[0])
                    except ValueError: continue
                    if orig in bdd_map:
                        ac = bdd_map[orig]; lines.append(f'{ac} ' + ' '.join(p[1:]))
                        if ac == 1: has_m = True
                        if ac == 4: has_b = True
                if not lines: continue
                stem = f'bdd_{img.stem}'
                link = di / (stem + img.suffix)
                if not link.exists(): _imglink(img, link)
                (dl / (stem + '.txt')).write_text('\n'.join(lines))
                nv += 1
                if has_m:
                    nm += 1
                    for k in range(2):
                        s2 = f'bdd_{img.stem}_m{k}'
                        ml = di / (s2 + img.suffix)
                        if not ml.exists(): _imglink(img, ml)
                        (dl / (s2 + '.txt')).write_text('\n'.join(lines))
                if has_b:
                    nb_cnt += 1
                    bl = di / (f'bdd_{img.stem}_b0' + img.suffix)
                    if not bl.exists(): _imglink(img, bl)
                    (dl / (f'bdd_{img.stem}_b0.txt')).write_text('\n'.join(lines))
            print(f'  BDD {split}: {nv} imgs | moto×3: {nm} | bike×2: {nb_cnt}')
        FLAG.touch(); print('BDD100K: done')
        # Free downloaded raw — hardlinks in MERGED are intact
        if str(BDD_DIR).startswith(str(WORK)):
            _shutil.rmtree(BDD_DIR, ignore_errors=True)
            print(f'  Freed {BDD_DIR}')

In [ ]:
# ── IDD (Indian Driving Dataset) → YOLO ──────────────────────────────────────
# motorcycle/autorickshaw x4, bicycle x2 oversample.
import xml.etree.ElementTree as ET
import os, random, yaml, subprocess
from pathlib import Path
from tqdm import tqdm

FLAG = OUT_DIR / '.idd_done'
if FLAG.exists():
    print('IDD: already done -- skipping')
else:
    IDD_NAME_MAP = {
        'car':0, 'auto':0, 'automobile':0,
        'autorickshaw':1, 'auto rickshaw':1, 'three wheeler':1,
        'motorcycle':1, 'motorbike':1, 'two wheeler':1, 'scooter':1,
        'bus':2,
        'truck':3, 'van':3, 'tempo':3, 'vehicle_fallback':3,
        'bicycle':4, 'cycle':4,
    }

    IDD_ROOT = None

    # ── 1. Known slugs (Kaggle mounts abhishekprajapat/idd-20k as idd-20k or idd_20k) ──
    for slug in ['idd-20k', 'idd_20k', 'idd-dataset-yolo-format', 'idd', 'idd-detection', 'idd20k', 'idd-yolo']:
        if (INPUT / slug).exists():
            IDD_ROOT = INPUT / slug
            print(f'IDD: found at /kaggle/input/{slug}/')
            break

    # ── 2. Search for idd20k_final by name (handles datasets/ merged folder) ──
    # When Kaggle combines user datasets under /kaggle/input/datasets/<user>/,
    # IDD_20K is at datasets/<user>/idd-20k/idd20k_final/
    if IDD_ROOT is None:
        hit = next(INPUT.rglob('idd20k_final'), None)
        if hit and hit.is_dir():
            IDD_ROOT = hit
            print(f'IDD: found idd20k_final at {hit}')

    # ── 3. Search all data.yaml files for IDD-specific class names ─────────────
    if IDD_ROOT is None:
        for yf in sorted(INPUT.rglob('data.yaml')):
            try:
                cfg = yaml.safe_load(yf.read_text())
                names_list = cfg.get('names', [])
                if isinstance(names_list, dict): names_list = list(names_list.values())
                idd_markers = {'autorickshaw', 'two wheeler', 'scooter', 'tempo', 'cycle'}
                if idd_markers & {n.strip().lower() for n in names_list}:
                    IDD_ROOT = yf.parent
                    print(f'IDD: found via data.yaml at {yf}')
                    break
            except Exception:
                pass

    # ── 4. Search for VOC Annotations/ dir ────────────────────────────────────
    if IDD_ROOT is None:
        for ann in INPUT.rglob('Annotations'):
            if ann.is_dir() and next(ann.glob('*.xml'), None):
                IDD_ROOT = ann.parent
                print(f'IDD: found VOC dataset at {IDD_ROOT}')
                break

    # ── 5. API download ────────────────────────────────────────────────────────
    if IDD_ROOT is None and KGL_USER:
        IDD_DL = WORK / '_idd_raw'
        IDD_DL.mkdir(parents=True, exist_ok=True)
        for slug in ['abhishekprajapat/idd-20k', 'ashwinak/idd-dataset-yolo-format']:
            r = subprocess.run(['kaggle', 'datasets', 'download', '-d', slug,
                                '-p', str(IDD_DL), '--unzip'],
                               capture_output=True, text=True)
            if r.returncode == 0:
                IDD_ROOT = IDD_DL
                print(f'IDD: downloaded via {slug}')
                break
            print(f'  {slug}: {r.stderr.strip()[:100]}')

    if IDD_ROOT is None:
        print('IDD: not found')
        mounted = [d.name for d in INPUT.iterdir() if d.is_dir()] if INPUT.exists() else []
        print(f'  Mounted top-level: {mounted}')
        print('  Attach abhishekprajapat/idd-20k as a Kaggle dataset and rerun.')
    else:
        # ── Detect format ─────────────────────────────────────────────────────
        yolo_yamls = list(IDD_ROOT.rglob('data.yaml'))
        yolo_idirs = list(IDD_ROOT.rglob('images'))
        yolo_ldirs = list(IDD_ROOT.rglob('labels'))
        voc_ann    = next(IDD_ROOT.rglob('Annotations'), None)
        voc_img    = next(IDD_ROOT.rglob('JPEGImages'), None)

        # Also check for images directly in train/ valid/ (no images/ subdir)
        direct_splits = []
        for split_kw, dst in [('train', 'train'), ('val', 'valid'), ('valid', 'valid')]:
            sdir = IDD_ROOT / split_kw
            if sdir.exists():
                imgs = list(sdir.glob('*.jpg')) + list(sdir.glob('*.png')) + list(sdir.glob('*.jpeg'))
                lbls = list(sdir.glob('*.txt'))
                if imgs and lbls:
                    direct_splits.append((dst, sdir, sdir))

        if yolo_idirs and yolo_ldirs:
            print('IDD: YOLO format (images/ + labels/ subdirs)')
            idd_map = {}
            if yolo_yamls:
                try:
                    cfg = yaml.safe_load(yolo_yamls[0].read_text())
                    names = cfg.get('names', [])
                    if isinstance(names, dict): names = [names[k] for k in sorted(names)]
                    idd_map = {i: IDD_NAME_MAP.get(n.strip().lower())
                               for i, n in enumerate(names)}
                    idd_map = {k: v for k, v in idd_map.items() if v is not None}
                    print(f'  class map (from {yolo_yamls[0].name}): {idd_map}')
                except Exception as e:
                    print(f'  data.yaml parse failed: {e}')
            if not idd_map:
                # Sample first label file to see what class IDs exist
                sample_lbl = next(IDD_ROOT.rglob('*.txt'), None)
                if sample_lbl:
                    ids = set()
                    for row in sample_lbl.read_text().strip().splitlines()[:20]:
                        p = row.split()
                        if p:
                            try: ids.add(int(p[0]))
                            except ValueError: pass
                    print(f'  No data.yaml -- sample class IDs from {sample_lbl.name}: {sorted(ids)}')
                    # Build map: anything in 0-4 stays, use IDD_NAME_MAP for remapping
                    idd_map = {i: i for i in range(10) if i <= 4}
                    print(f'  class map: identity 0-4 (set data.yaml for correct mapping)')

            for kw, dst in [('train', 'train'), ('val', 'valid'), ('test', 'valid')]:
                idirs = [d for d in yolo_idirs if kw in str(d).lower()]
                ldirs = [d for d in yolo_ldirs if kw in str(d).lower()]
                if not idirs: continue
                idir = idirs[0]
                # Try 3 label dir patterns: from rglob, labels/<split>/, <split>/labels/
                if ldirs:
                    ldir = ldirs[0]
                elif (IDD_ROOT / 'labels' / kw).exists():
                    ldir = IDD_ROOT / 'labels' / kw
                else:
                    ldir = idir.parent / 'labels'
                print(f'  IDD {kw}: images={idir.relative_to(INPUT)} labels={ldir.relative_to(INPUT) if ldir.exists() else ldir}')
                di = MERGED / dst / 'images'
                dl = MERGED / dst / 'labels'
                nv = nm = nb = 0
                for img in tqdm(list(idir.glob('*.*')), desc=f'IDD {kw}'):
                    lp = ldir / (img.stem + '.txt')
                    if not lp.exists(): continue
                    lines, has_m, has_b = [], False, False
                    for row in lp.read_text().strip().splitlines():
                        p = row.split()
                        if not p: continue
                        try: orig = int(p[0])
                        except ValueError: continue
                        if orig in idd_map:
                            ac = idd_map[orig]
                            lines.append(f'{ac} ' + ' '.join(p[1:]))
                            if ac == 1: has_m = True
                            if ac == 4: has_b = True
                    if not lines: continue
                    stem = f'idd_{kw}_{img.stem}'
                    link = di / (stem + img.suffix)
                    if not link.exists(): os.symlink(img.resolve(), link)
                    (dl / (stem + '.txt')).write_text('\n'.join(lines))
                    nv += 1
                    if has_m:
                        nm += 1
                        for k in range(3):  # x4 total
                            mlink = di / (f'idd_{kw}_{img.stem}_m{k}' + img.suffix)
                            if not mlink.exists(): os.symlink(img.resolve(), mlink)
                            (dl / (f'idd_{kw}_{img.stem}_m{k}.txt')).write_text('\n'.join(lines))
                    if has_b:
                        nb += 1   # x2 total
                        blink = di / (f'idd_{kw}_{img.stem}_b0' + img.suffix)
                        if not blink.exists(): os.symlink(img.resolve(), blink)
                        (dl / (f'idd_{kw}_{img.stem}_b0.txt')).write_text('\n'.join(lines))
                print(f'  IDD {kw}: {nv} imgs | moto x4: {nm} | bike x2: {nb}')
                if nv == 0 and list(idir.glob('*.*')):
                    sample = next(idir.glob('*.*'))
                    slp = ldir / (sample.stem + '.txt')
                    print(f'  DEBUG: sample img={sample.name} label_exists={slp.exists()}')
                    if slp.exists():
                        print(f'  DEBUG: label content: {slp.read_text()[:200]}')

        elif direct_splits:
            print('IDD: YOLO format (images directly in train/val dirs)')
            idd_map = {i: i for i in range(10) if i <= 4}
            for dst, idir, ldir in direct_splits:
                di = MERGED / dst / 'images'
                dl = MERGED / dst / 'labels'
                nv = nm = nb = 0
                for img in tqdm(list(idir.glob('*.jpg')) + list(idir.glob('*.png')),
                                desc=f'IDD direct {dst}'):
                    lp = ldir / (img.stem + '.txt')
                    if not lp.exists(): continue
                    lines, has_m, has_b = [], False, False
                    for row in lp.read_text().strip().splitlines():
                        p = row.split()
                        if not p: continue
                        try: orig = int(p[0])
                        except ValueError: continue
                        if orig in idd_map:
                            ac = idd_map[orig]
                            lines.append(f'{ac} ' + ' '.join(p[1:]))
                            if ac == 1: has_m = True
                            if ac == 4: has_b = True
                    if not lines: continue
                    stem = f'idd_{dst}_{img.stem}'
                    link = di / (stem + img.suffix)
                    if not link.exists(): os.symlink(img.resolve(), link)
                    (dl / (stem + '.txt')).write_text('\n'.join(lines))
                    nv += 1
                    if has_m:
                        nm += 1
                        for k in range(3):
                            mlink = di / (f'idd_{dst}_{img.stem}_m{k}' + img.suffix)
                            if not mlink.exists(): os.symlink(img.resolve(), mlink)
                            (dl / (f'idd_{dst}_{img.stem}_m{k}.txt')).write_text('\n'.join(lines))
                    if has_b:
                        nb += 1
                        blink = di / (f'idd_{dst}_{img.stem}_b0' + img.suffix)
                        if not blink.exists(): os.symlink(img.resolve(), blink)
                        (dl / (f'idd_{dst}_{img.stem}_b0.txt')).write_text('\n'.join(lines))
                print(f'  IDD direct {dst}: {nv} imgs | moto x4: {nm} | bike x2: {nb}')

        elif voc_ann and voc_img:
            print('IDD: VOC XML format')
            xmls = list(voc_ann.glob('*.xml'))
            random.seed(42); random.shuffle(xmls)
            idx = int(len(xmls) * 0.9)
            for dst, xlist in [('train', xmls[:idx]), ('valid', xmls[idx:])]:
                di = MERGED / dst / 'images'
                dl = MERGED / dst / 'labels'
                nv = nm = nb = 0
                for xp in tqdm(xlist, desc=f'IDD VOC {dst}'):
                    try: root_el = ET.parse(xp).getroot()
                    except ET.ParseError: continue
                    sz = root_el.find('size')
                    if sz is None or sz.find('width') is None: continue
                    W, H = int(sz.find('width').text), int(sz.find('height').text)
                    if W <= 0 or H <= 0: continue
                    lines, has_m, has_b = [], False, False
                    for obj in root_el.findall('object'):
                        nm_raw = obj.find('name').text.strip()
                        ac = IDD_NAME_MAP.get(nm_raw.lower())
                        if ac is None: continue
                        bb = obj.find('bndbox')
                        x1, y1, x2, y2 = (float(bb.find(t).text)
                                           for t in ['xmin', 'ymin', 'xmax', 'ymax'])
                        lines.append(
                            f'{ac} {max(.001, min(.999, (x1+x2)/2/W)):.6f}'
                            f' {max(.001, min(.999, (y1+y2)/2/H)):.6f}'
                            f' {max(.001, min(.999, (x2-x1)/W)):.6f}'
                            f' {max(.001, min(.999, (y2-y1)/H)):.6f}')
                        if ac == 1: has_m = True
                        if ac == 4: has_b = True
                    if not lines: continue
                    ip = next((voc_img / (xp.stem + e) for e in ['.jpg', '.jpeg', '.png']
                               if (voc_img / (xp.stem + e)).exists()), None)
                    if ip is None: continue
                    stem = f'idd_{xp.stem}'
                    link = di / (stem + ip.suffix)
                    if not link.exists(): os.symlink(ip.resolve(), link)
                    (dl / (stem + '.txt')).write_text('\n'.join(lines))
                    nv += 1
                    if has_m:
                        nm += 1
                        for k in range(3):
                            mlink = di / (f'idd_{xp.stem}_m{k}' + ip.suffix)
                            if not mlink.exists(): os.symlink(ip.resolve(), mlink)
                            (dl / (f'idd_{xp.stem}_m{k}.txt')).write_text('\n'.join(lines))
                    if has_b:
                        nb += 1
                        blink = di / (f'idd_{xp.stem}_b0' + ip.suffix)
                        if not blink.exists(): os.symlink(ip.resolve(), blink)
                        (dl / (f'idd_{xp.stem}_b0.txt')).write_text('\n'.join(lines))
                print(f'  IDD VOC {dst}: {nv} | moto x4: {nm} | bike x2: {nb}')
        else:
            print(f'IDD: unrecognised structure at {IDD_ROOT}')
            print(f'  rglob images/: {yolo_idirs[:3]}')
            print(f'  rglob labels/: {yolo_ldirs[:3]}')
            print(f'  direct splits: {direct_splits}')
        FLAG.touch()
        print('IDD: done')

In [ ]:
# ── KITTI — skipped in S2 ────────────────────────────────────────────────────
# KITTI (~12 GB zip + 12 GB extracted = 24 GB peak) exceeds Kaggle's 20 GB
# working-disk quota during extraction. S1 already trained on KITTI.
# BDD100K (70k) + IDD (20k Indian) covers S2 domain adaptation.
# Re-add in S3 by attaching a pre-converted Kaggle KITTI YOLO dataset.
print('KITTI: skipped in S2 (disk budget — S1 already included it)')

In [ ]:
# ── Dataset stats + data.yaml ────────────────────────────────────────────────
import shutil
from pathlib import Path
from collections import Counter

def _count(split):
    ldir = MERGED/split/'labels'
    imgs = list((MERGED/split/'images').glob('*.*'))
    stats = Counter()
    for lp in ldir.glob('*.txt'):
        for row in lp.read_text().splitlines():
            p = row.strip().split()
            if p:
                try: stats[int(p[0])] += 1
                except ValueError: pass
    return imgs, stats

print('='*60)
for split in ['train','valid']:
    imgs, stats = _count(split)
    total = sum(stats.values())
    bad = {k:v for k,v in stats.items() if k<0 or k>=NC}
    print(f'\n{split}: {len(imgs):,} images | {total:,} boxes')
    for cid, name in enumerate(CLASSES):
        bar = '█' * min(40, int(40*stats.get(cid,0)/max(total,1)))
        print(f'  {cid} {name:12s}: {stats.get(cid,0):8,}  {bar}')
    if bad:
        print(f'  ⚠  OUT-OF-RANGE IDs: {bad}')
    else:
        print(f'  ✓ all IDs in [0,{NC-1}]')
print('='*60)

# Per-source breakdown
print('\nSource breakdown (train images):')
sources = Counter()
for p in (MERGED/'train'/'images').glob('*.*'):
    prefix = p.stem.split('_')[0]
    sources[prefix] += 1
for src, n in sorted(sources.items(), key=lambda x: -x[1]):
    print(f'  {src:12s}: {n:,}')

# Disk usage
total_gb, used_gb, free_gb = (x/1024**3 for x in shutil.disk_usage('/kaggle/working'))
print(f'\nDisk /kaggle/working: {used_gb:.1f} GB used / {total_gb:.1f} GB total ({free_gb:.1f} GB free)')

n_train = len(list((MERGED/'train'/'images').glob('*.*')))
if n_train < 500:
    raise AssertionError(
        f'Only {n_train} training images found.\n'
        'BDD100K and UA-DETRAC likely failed to load.\n'
        'Fix: set KAGGLE_USERNAME + KAGGLE_KEY in Secrets, OR attach BDD100K as a dataset.')
elif n_train < 5_000:
    print(f'\n⚠  Only {n_train} images — training will run but accuracy will be limited.')
    print('   Add BDD100K (attach dataset or set Kaggle Secrets) for full training.')
else:
    print(f'\n✓ {n_train:,} training images ready')

yaml_path.write_text(
    f'path: {MERGED.resolve()}\ntrain: train/images\nval:   valid/images\n'
    f'\nnc: {NC}\nnames: {CLASSES}\n')
print(f'data.yaml → {yaml_path}')

In [ ]:
# ── Phase B — Full fine-tune on merged dataset (20 epochs, ~17 h) ────────────
# S1 only completed Phase A (frozen backbone, 11 epochs → mAP50 0.659).
# Phase B unlocks the full network — this is the most important step.
#
# lr0=2e-4: moderate — Phase A warm-up already moved backbone weights
#           well away from the COCO init; no need for high LR.
# lrf=0.05: cosine decay to 1e-5 — allows slow convergence at end.
# box=9.0/cls=0.3: prioritise tight localisation over class scores;
#                  critical for TTC accuracy downstream.
# copy_paste=0.3: pastes cut-out vehicles onto scenes — best aug for
#                 rare-class (motorcycle, bicycle) recall.
# close_mosaic=8: last 8 epochs train without mosaic → smoother confidence.
# patience=20: allow full 20 epochs; validation mAP improves slowly
#              after a frozen-backbone checkpoint.
from ultralytics import YOLO
import time

assert UPLOADED_S1.exists(), (
    'argus_s1.pt not found at /kaggle/working/\n'
    'Download runs/s1a/argus_s1a/weights/best.pt from Session 1 output and upload.')
assert yaml_path.exists(), 'Run the dataset stats cell first'

t0 = time.time()
if PHB_LAST.exists():
    print(f'Phase B: resuming from {PHB_LAST}')
    model = YOLO(str(PHB_LAST))
    model.train(resume=True)
else:
    print(f'Phase B: starting from {UPLOADED_S1}  (mAP50 baseline: 0.659)')
    model = YOLO(str(UPLOADED_S1))
    model.train(
        data=str(yaml_path), imgsz=IMGSZ, epochs=20, batch=BATCH, device=DEVICE,
        project=str(RUNS_DIR/'s2_phb'), name='argus_s2_phb', exist_ok=True,
        freeze=0, optimizer='AdamW',
        lr0=2e-4, lrf=0.05, warmup_epochs=1,
        weight_decay=5e-4,
        box=9.0, cls=0.3, dfl=2.0,
        mosaic=0.5, mixup=0.0, copy_paste=0.3, erasing=0.2,
        fliplr=0.5, scale=0.5, degrees=0.0,
        close_mosaic=8,
        save_period=5, patience=20, amp=True, verbose=True,
    )

elapsed = (time.time() - t0) / 3600
print(f'\nPhase B done in {elapsed:.1f} h')
if PHB_BEST.exists():
    print(f'Phase B best: {PHB_BEST}')
    print('Expected mAP50: 0.76–0.82 | Expected mAP50-95: 0.50–0.56')

In [ ]:
# ── Proximity-aware hard-example dataset builder ──────────────────────────────
# Scoring dimensions:
#   density  : penalises sparse scenes (min(n/5,3)×5, capped at 15)
#   diversity: classes present (×2 per unique class)
#   moto     : motorcycle present (+4) — top-priority near-miss actor
#   prox     : nearby vehicle pairs within 0.15 normalised distance (×3, cap 30)
#   mid_small: 0.03 < w or h < 0.12 — hard detection size range (×2)
import os, shutil, statistics
from pathlib import Path
from tqdm import tqdm

def _score(lp: Path) -> float:
    rows = []
    for line in lp.read_text().strip().splitlines():
        p = line.split()
        if len(p) < 5: continue
        try: rows.append((int(p[0]), float(p[1]), float(p[2]), float(p[3]), float(p[4])))
        except (ValueError, IndexError): continue
    if not rows: return 0.0
    classes  = [r[0] for r in rows]
    density  = min(len(rows)/5.0, 3.0) * 5
    diversity = len(set(classes)) * 2
    moto     = 4 if 1 in classes else 0
    prox     = min(
        sum(3 for i in range(len(rows)) for j in range(i+1, len(rows))
            if ((rows[i][1]-rows[j][1])**2 + (rows[i][2]-rows[j][2])**2)**0.5 < 0.15),
        30)
    mid_small = sum(2 for r in rows if 0.03 < r[3] < 0.12 or 0.03 < r[4] < 0.12)
    return density + diversity + moto + prox + mid_small

src_img = MERGED/'train'/'images'
src_lbl = MERGED/'train'/'labels'

print('Scoring training images...')
scored = [(s, lp) for lp in tqdm(list(src_lbl.glob('*.txt')))
          if (s := _score(lp)) > 0]
scored.sort(key=lambda x: x[0], reverse=True)

N_TRAIN = min(50_000, max(10_000, int(len(scored)*0.70)))
N_VAL   = min(4_000, len(scored)-N_TRAIN)
hard_train = [lp for _,lp in scored[:N_TRAIN]]
hard_val   = [lp for _,lp in scored[N_TRAIN:N_TRAIN+N_VAL]]

all_scores = [s for s,_ in scored]
print(f'Selected {len(hard_train):,} train + {len(hard_val):,} val')
print(f'Score: max={max(all_scores):.0f}  mean={statistics.mean(all_scores):.1f}'
      f'  cutoff={scored[N_TRAIN-1][0]:.0f}')

for dst, llist in [('train',hard_train),('valid',hard_val)]:
    hi = HARD_DIR/dst/'images'; hl = HARD_DIR/dst/'labels'
    hi.mkdir(parents=True, exist_ok=True); hl.mkdir(parents=True, exist_ok=True)
    for lp in tqdm(llist, desc=f'Link {dst}'):
        ip = next((src_img/(lp.stem+e) for e in ['.jpg','.jpeg','.png']
                   if (src_img/(lp.stem+e)).exists()), None)
        if ip:
            link = hi/ip.name
            if not link.exists(): os.symlink(ip.resolve(), link)
            shutil.copy2(lp, hl/lp.name)

(HARD_DIR/'data.yaml').write_text(
    f'path: {HARD_DIR.resolve()}\ntrain: train/images\nval:   valid/images\n'
    f'\nnc: {NC}\nnames: {CLASSES}\n')
print(f'Hard dataset: {len(list((HARD_DIR/"train"/"images").glob("*.*"))):,} train images')

In [ ]:
# ── Phase HM-A — 6 epochs, high LR on hard subset ───────────────────────────
# Starts from Phase B best (full-network fine-tune) — NOT from raw S1 checkpoint.
# lr0=3e-4: 1.5× Phase B peak — aggressive on hard cases without catastrophic
#           forgetting, since Phase B has already established strong priors.
# dfl=2.0:  sharpens distribution focal peaks → better small-vehicle boxes.
# copy_paste=0.3: still needed — hard set skews toward dense/occluded scenes.
# patience=0: run all 6 epochs; hard set is smaller so variance is higher.
from ultralytics import YOLO
import shutil, time

hard_yaml = HARD_DIR / 'data.yaml'
assert hard_yaml.exists(), 'Run the hard-example builder cell first'

start = PHB_BEST if PHB_BEST.exists() else UPLOADED_S1
if not start.exists():
    raise FileNotFoundError(
        f'{start} not found.\n'
        'Run Phase B cell first, or upload argus_s1.pt for a weaker baseline.')

t0 = time.time()
if HMA_LAST.exists():
    print(f'HM-A: resuming from {HMA_LAST}')
    model = YOLO(str(HMA_LAST)); model.train(resume=True)
else:
    print(f'HM-A: starting from {start}')
    model = YOLO(str(start))
    model.train(
        data=str(hard_yaml), imgsz=IMGSZ, epochs=6, batch=BATCH, device=DEVICE,
        project=str(RUNS_DIR/'s2_hma'), name='argus_s2_hma', exist_ok=True,
        freeze=0, optimizer='AdamW', lr0=3e-4, lrf=0.01, warmup_epochs=0,
        weight_decay=5e-4, box=9.0, cls=0.3, dfl=2.0,
        mosaic=0.4, mixup=0.0, copy_paste=0.3, erasing=0.3,
        fliplr=0.5, scale=0.8, perspective=0.001,
        close_mosaic=2,
        save_period=3, patience=0, amp=True, verbose=True,
    )

print(f'HM-A done in {(time.time()-t0)/3600:.1f} h')
if HMA_BEST.exists(): print(f'HM-A best: {HMA_BEST}')

In [ ]:
# ── Mid-training re-scoring — refresh bottom 10% of hard set ─────────────────
# After Phase A the model has learned the current hard set well.
# We replace the lowest-scoring 10% with fresh hard images it hasn't seen,
# forcing it to generalise rather than memorise.
import os, shutil, statistics
from pathlib import Path
from tqdm import tqdm

REFRESH_FRAC = 0.10

def _score(lp: Path) -> float:
    rows = []
    for line in lp.read_text().strip().splitlines():
        p = line.split()
        if len(p) < 5: continue
        try: rows.append((int(p[0]), float(p[1]), float(p[2]), float(p[3]), float(p[4])))
        except (ValueError, IndexError): continue
    if not rows: return 0.0
    classes  = [r[0] for r in rows]
    density  = min(len(rows)/5.0, 3.0) * 5
    diversity = len(set(classes)) * 2
    moto     = 4 if 1 in classes else 0
    prox     = min(
        sum(3 for i in range(len(rows)) for j in range(i+1, len(rows))
            if ((rows[i][1]-rows[j][1])**2 + (rows[i][2]-rows[j][2])**2)**0.5 < 0.15),
        30)
    mid_small = sum(2 for r in rows if 0.03 < r[3] < 0.12 or 0.03 < r[4] < 0.12)
    return density + diversity + moto + prox + mid_small

src_img = MERGED / 'train' / 'images'
src_lbl = MERGED / 'train' / 'labels'
hard_lbl = HARD_DIR / 'train' / 'labels'
hard_img = HARD_DIR / 'train' / 'images'

# Score all images in the current hard training set
print('Re-scoring current hard training set...')
current = [(lp, _score(lp)) for lp in tqdm(list(hard_lbl.glob('*.txt')))]
current.sort(key=lambda x: x[1])  # ascending — worst first

n_replace = int(len(current) * REFRESH_FRAC)
to_remove = set(lp.stem for lp, _ in current[:n_replace])

# Score all full-merged images not already in hard set
hard_stems = {lp.stem for lp, _ in current}
print(f'Scoring {len(list(src_lbl.glob("*.txt"))):,} full-set images for replacements...')
candidates = [(lp, _score(lp)) for lp in tqdm(list(src_lbl.glob('*.txt')))
              if lp.stem not in hard_stems and _score(lp) > 0]
candidates.sort(key=lambda x: x[1], reverse=True)  # descending — best first
replacements = candidates[:n_replace]

print(f'Replacing {n_replace} bottom images with {len(replacements)} fresh hard images')

# Remove bottom scorers
for stem in to_remove:
    (hard_lbl / (stem + '.txt')).unlink(missing_ok=True)
    for ext in ['.jpg', '.jpeg', '.png']:
        p = hard_img / (stem + ext)
        if p.exists():
            p.unlink(); break

# Add replacements (symlinks for images, copy for labels)
for lp, score in tqdm(replacements, desc='Adding replacements'):
    ip = next((src_img/(lp.stem+e) for e in ['.jpg','.jpeg','.png']
               if (src_img/(lp.stem+e)).exists()), None)
    if ip:
        link = hard_img / ip.name
        if not link.exists():
            os.symlink(ip.resolve(), link)
        shutil.copy2(lp, hard_lbl / lp.name)

new_count = len(list(hard_lbl.glob('*.txt')))
print(f'Hard training set after refresh: {new_count:,} images')

# Write hard_v2.yaml (same path — Phase B reads it)
(HARD_DIR/'data.yaml').write_text(
    f'path: {HARD_DIR.resolve()}\ntrain: train/images\nval:   valid/images\n'
    f'\nnc: {NC}\nnames: {CLASSES}\n')
print('hard_v2.yaml written — ready for Phase B')


In [ ]:
# ── Phase HM-B — 6 epochs, reduced LR on refreshed hard set ─────────────────
# lr0=5e-5: 6× lower than HM-A — consolidation rather than re-learning.
# close_mosaic=4: last 4 epochs on clean images → smoother val confidence.
# patience=10: allow early stop if generalisation plateaus.
from ultralytics import YOLO
import shutil, time

hard_yaml = HARD_DIR / 'data.yaml'
assert hard_yaml.exists(), 'Run re-scoring cell first'
start = HMA_BEST if HMA_BEST.exists() else (PHB_BEST if PHB_BEST.exists() else UPLOADED_S1)
assert start.exists(), f'No HM-A checkpoint found — run HM-A cell first'

t0 = time.time()
if HMB_LAST.exists():
    print(f'HM-B: resuming from {HMB_LAST}')
    model = YOLO(str(HMB_LAST)); model.train(resume=True)
else:
    print(f'HM-B: starting from {start}')
    model = YOLO(str(start))
    model.train(
        data=str(hard_yaml), imgsz=IMGSZ, epochs=6, batch=BATCH, device=DEVICE,
        project=str(RUNS_DIR/'s2_hmb'), name='argus_s2_hmb', exist_ok=True,
        freeze=0, optimizer='AdamW', lr0=5e-5, lrf=0.01, warmup_epochs=0,
        weight_decay=5e-4, box=9.0, cls=0.3, dfl=2.0,
        mosaic=0.4, mixup=0.0, copy_paste=0.2, erasing=0.2,
        fliplr=0.5, scale=0.6,
        close_mosaic=4,
        save_period=3, patience=10, amp=True, verbose=True,
    )

elapsed = (time.time() - t0) / 3600
print(f'HM-B done in {elapsed:.1f} h')

# Save best checkpoint as argus_s2.pt
best = next((c for c in [HMB_BEST, HMA_BEST, PHB_BEST] if c.exists()), None)
if best:
    shutil.copy2(best, WORK/'argus_s2.pt')
    print(f'\n✓ Saved → {WORK}/argus_s2.pt')
    print('  Download from Output tab → upload as argus_s2.pt for Session 3')

In [ ]:
# ── Validation (conf=0.001, iou=0.6) ─────────────────────────────────────────
from ultralytics import YOLO

candidates = [WORK/'argus_s2.pt', HMB_BEST, HMA_BEST, PHB_BEST]
EVAL = next((c for c in candidates if c.exists()), None)
assert EVAL, 'No S2 model found — run training cells first'
print(f'Evaluating: {EVAL}')

model = YOLO(str(EVAL))
metrics = model.val(
    data=str(yaml_path), imgsz=IMGSZ, batch=BATCH, device=DEVICE,
    conf=0.001, iou=0.6, verbose=True,
)

print('\n' + '='*60)
print('  ARGUS S2 Validation Results')
print('='*60)
print(f'  mAP50    : {metrics.box.map50:.4f}')
print(f'  mAP50-95 : {metrics.box.map:.4f}  ← PRIMARY')
print()
for name, ap50, ap in zip(CLASSES, metrics.box.ap50, metrics.box.ap):
    flag = '  ← near-miss actor' if name == 'motorcycle' else ''
    print(f'  {name:12s}: AP50={ap50:.3f}  AP50-95={ap:.3f}{flag}')
print('='*60)
if metrics.box.map < 0.52:
    print('\n⚠  mAP50-95 < 0.52 — check data pipeline and re-run')
elif metrics.box.map < 0.60:
    print(f'\n→  mAP50-95 {metrics.box.map:.3f} — S3 polish should push above 0.62')
else:
    print(f'\n✓  mAP50-95 {metrics.box.map:.3f} — strong S2 baseline for S3')